## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

## Load and check data

In [3]:
df = pd.read_csv('../data/cleaned/books_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (175, 8)

Columns: ['title', 'category', 'book_url', 'image_url', 'description', 'image_path', 'title_length', 'description_length']

First few rows:


,title,category,book_url,image_url,description,image_path,title_length,description_length
0,Soumission,Fiction,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...,"Dans une France assez proche de la nôtre, un h...",images/book_0.jpg,10,1093
1,Private Paris (Private #10),Fiction,https://books.toscrape.com/catalogue/private-p...,https://books.toscrape.com/media/cache/9d/05/9...,Paris is burning--and only Private's Jack Morg...,images/book_1.jpg,27,1131
2,"We Love You, Charlie Freeman",Fiction,https://books.toscrape.com/catalogue/we-love-y...,https://books.toscrape.com/media/cache/5f/15/5...,"The Freeman family--Charles, Laurel, and their...",images/book_2.jpg,28,1491
3,Thirst,Fiction,https://books.toscrape.com/catalogue/thirst_94...,https://books.toscrape.com/media/cache/c4/0a/c...,"On a searing summer Friday, Eddie Chapman has ...",images/book_3.jpg,6,1451
4,The Murder That Never Was (Forensic Instincts #5),Fiction,https://books.toscrape.com/catalogue/the-murde...,https://books.toscrape.com/media/cache/dc/44/d...,"Given the opportunity, would you assume someon...",images/book_4.jpg,49,1843


In [4]:
print("Class Distribution:")
print(df['category'].value_counts())
print("\nPercentage:")
print(df['category'].value_counts(normalize=True) * 100)

Class Distribution:
category
Nonfiction    110
Fiction        65
Name: count, dtype: int64

Percentage:
category
Nonfiction    62.857143
Fiction       37.142857
Name: proportion, dtype: float64


## Preparing Data. For our case, using description as feature

In [5]:
x = df['description']
y = df['category']

In [6]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Classes: {label_encoder.classes_}")
print(f"Feature shape: {x.shape}")
print(f"Target shape: {y_encoded.shape}")

Classes: ['Fiction' 'Nonfiction']
Feature shape: (175,)
Target shape: (175,)


## Preparing train-test split

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set size: {len(x_train)}")
print(f"Test set size: {len(x_test)}")
print(f"\nTraining set distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nTest set distribution:")
print(pd.Series(y_test).value_counts())

Training set size: 140
Test set size: 35

Training set distribution:
1    88
0    52
Name: count, dtype: int64

Test set distribution:
1    22
0    13
Name: count, dtype: int64


## TF-IDF feature extraction

In [8]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=100,
    min_df=2,
    max_df=0.8,
    ngram_range=(1,2),
    stop_words='english'
)

x_train_tfidf=tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf=tfidf_vectorizer.transform(x_test)

print(f"Training features shape: {x_train_tfidf.shape}")
print(f"Test features shape: {x_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

Training features shape: (140, 100)
Test features shape: (35, 100)
Vocabulary size: 100


## Train Multinomial Naive Bayes

In [9]:
nb_model = MultinomialNB()
nb_model.fit(x_train_tfidf, y_train)

y_pred_nb = nb_model.predict(x_test_tfidf)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_nb):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_nb):.4f}")

print("\nclassification report:")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=label_encoder.classes_
))


Accuracy: 0.6857
Precision: 0.7037
Recall: 0.8636
F1-score: 0.7755

classification report:
              precision    recall  f1-score   support

     Fiction       0.62      0.38      0.48        13
  Nonfiction       0.70      0.86      0.78        22

    accuracy                           0.69        35
   macro avg       0.66      0.62      0.63        35
weighted avg       0.67      0.69      0.66        35



## Train Logistic Regression

In [10]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(x_train_tfidf, y_train)

y_pred_lr = lr_model.predict(x_test_tfidf)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")

print("\nclassification report:")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=label_encoder.classes_
))


Accuracy: 0.7143
Precision: 0.7000
Recall: 0.9545
F1-Score: 0.8077

classification report:
              precision    recall  f1-score   support

     Fiction       0.80      0.31      0.44        13
  Nonfiction       0.70      0.95      0.81        22

    accuracy                           0.71        35
   macro avg       0.75      0.63      0.63        35
weighted avg       0.74      0.71      0.67        35



## Train Random Forest

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

rf_model.fit(x_train_tfidf, y_train)

y_pred_rf = rf_model.predict(x_test_tfidf)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=label_encoder.classes_
))


Accuracy: 0.7429
Precision: 0.7407
Recall: 0.9091
F1-Score: 0.8163

Classification Report:
              precision    recall  f1-score   support

     Fiction       0.75      0.46      0.57        13
  Nonfiction       0.74      0.91      0.82        22

    accuracy                           0.74        35
   macro avg       0.75      0.69      0.69        35
weighted avg       0.74      0.74      0.73        35



## Train Support Vector Machine

In [ ]:
svm_model = SVC(
    kernel = 'linear'
    random_state = 